In [2]:
%cd  d:\\DoAn\\Video_Anomaly_Detection

d:\DoAn\Video_Anomaly_Detection


In [ ]:
import re
from Web.src.utils.misc import draw_anomaly_graph,  smooth_filter , find_anomaly_regions
from AI.src.data.dataset import VADFrameLevelDataset
import os


Initializing DLL path for Windows


In [4]:
def parse_pred_file(file_path):
    """
    Parse the prediction result file that contains scores and ground truth labels
    
    Returns:
        all_scores: List of prediction score lists
        all_anomaly_ranges: List of lists of anomaly regions (start, end) for each video
        video_indices: List of video indices
    """
    all_scores = []
    all_labels = []
    all_anomaly_ranges = []
    video_indices = []
    
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find all instances of the pattern [scores],[labels],index
    pattern = r'\[(.*?)\],\[(.*?)\],(\d+)'
    matches = re.findall(pattern, content)
    
    for match in matches:
        scores_str, labels_str, video_idx = match
        
        # Parse scores
        scores = [float(s.strip()) for s in scores_str.split(',')]
        
        # Parse labels
        labels = [int(l.strip()) for l in labels_str.split(',')]
        
        # Find anomaly regions from labels
        anomaly_ranges = []
        start = None
        
        for i, label in enumerate(labels):
            if label == 1 and start is None:
                start = i
            elif label == 0 and start is not None:
                anomaly_ranges.append((start, i - 1))
                start = None
        
        # If there's an anomaly that extends to the end of the video
        if start is not None:
            anomaly_ranges.append((start, len(labels) - 1))
        
        all_scores.append(scores)
        all_labels.append(labels)
        all_anomaly_ranges.append(anomaly_ranges)
        video_indices.append(int(video_idx))
    
    return all_scores, all_anomaly_ranges, video_indices

In [ ]:
dataset_root = "d:/DoAn/Video_Anomaly_Detection/data/test"  # Adjust to your Windows path
annotation_file = "label.csv"

dataset = VADFrameLevelDataset(
    root = dataset_root,
    annotation=annotation_file,
    load="v4"
)
video_paths = dataset._VADFrameLevelDataset__annotation["path"].tolist()
video_names = {i: os.path.splitext(os.path.basename(path))[0] for i, path in enumerate(video_paths)}

##fig1

In [ ]:
video_scores,labels,video_indices = parse_pred_file(r"D:\DoAn\Video_Anomaly_Detection\pred_result.txt")
output_dir = "D:/DoAn/Video_Anomaly_Detection/plots"
os.makedirs(output_dir, exist_ok=True)

for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]

    video_name = video_names.get(video_idx, f"Video_{video_idx}")

    save_path = os.path.join("output_dir",f"fig1_{video_name}.png")

    draw_anomaly_graph(pred=pred,
                       anomaly_ranges=video_anomaly_ranges,
                       video_name=video_name,
                       save_path=save_path,)

##fig2

In [ ]:
# Thông số cho phát hiện peak
window_length = 15
polyorder = 6
height = 0.7
prominence = 0.3

# Tạo thư mục để lưu biểu đồ
output_dir = "D:/DoAn/Video_Anomaly_Detection/plots"
os.makedirs(output_dir, exist_ok=True)


for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]
    video_name = video_names.get(video_idx, f"Video_{video_idx}")
    
    print(f"\nProcessing video: {video_name} (index: {video_idx})")
    
    # 1. Smooth the data
    smoothed_pred = smooth_filter(pred, window_length=window_length, polyorder=polyorder)
    
    # 2. Find anomaly regions using the utility function
    detected_regions, processed_signal, peaks = find_anomaly_regions(
        pred, 
        high_threshold=height,
        low_threshold=prominence,
        MERGE_GAP=5
    )
    
    # Print peak information
    print(f"Video {video_name} has {len(peaks)} detected peaks")
    if len(peaks) > 0:
        print(f"Peaks at frames: {peaks}")
        print(f"Detected anomaly regions: {detected_regions}")
    
    # 3. Draw graph with ground truth and detected anomaly regions
    save_path = os.path.join(output_dir, f"fig2_{video_name}.png")
    
    draw_anomaly_graph(
        preds=pred,
        anomaly_ranges=video_anomaly_ranges,
        video_name=video_name,
        save_path=save_path,
        smooth_pred=smoothed_pred,
        smooth_label="Smoothed pred",
        additional_anomaly_ranges=detected_regions,
        additional_anomaly_color="green"
    )